# Exploring Relevant Subreddits

In [2]:
import pandas as pd

## Load Sample Dataset

In [3]:
SAMPLE_POSTS_PATH = "data/prelim_historical_10000_11_11_26.csv"

In [4]:
df_sample_posts = pd.read_csv(SAMPLE_POSTS_PATH)

## Explore Dataset

In [5]:
df_sample_posts["id"].nunique()

10183

In [6]:
df_sample_posts["subreddit"].nunique()

1

In [28]:
df_sample_posts.columns

Index(['id', 'subreddit', 'title', 'selftext', 'url', 'score', 'num_comments',
       'flair', 'created_utc'],
      dtype='object')

### Exploring Title

In [31]:
titles = df_sample_posts["title"]
titles.head(2)

0    Need help with resume
1    Seeking resume advice
Name: title, dtype: object

In [ ]:
titles.

In [7]:
has_resume = df_sample_posts["title"].str.contains("resume")

In [8]:
df_sample_posts[has_resume]["title"].nunique()

2005

In [9]:
removed_text = df_sample_posts[has_resume]["selftext"] == "[removed]"
removed_text.value_counts()

selftext
True     1342
False     882
Name: count, dtype: int64

In [10]:
deleted_text = df_sample_posts[has_resume]["selftext"] == "[deleted]"
deleted_text.value_counts()

selftext
False    2220
True        4
Name: count, dtype: int64

In [11]:
relevant_cohort = df_sample_posts[
    has_resume & ~removed_text & ~deleted_text
].dropna()

In [17]:
relevant_cohort["num_comments"].describe()

count    654.000000
mean       9.304281
std       19.863564
min        0.000000
25%        3.000000
50%        5.000000
75%        9.000000
max      271.000000
Name: num_comments, dtype: float64

In [19]:
sample_url = relevant_cohort.iloc[0]["url"]

In [20]:
import os
import requests

def download_reddit_image(url: str, save_path: str) -> bool:
    """
    Downloads an image from a Reddit image URL (e.g. https://i.redd.it/...).
    """
    # Reddit blocks default python requests, so a custom User-Agent is required
    headers = {"User-Agent": "Mozilla/5.0"}
    
    # Ensure destination folder exists
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    
    response = requests.get(url, headers=headers, timeout=15)
    if response.status_code == 200:
        with open(save_path, "wb") as f:
            f.write(response.content)
        return True
    else:
        print(f"Failed to download {url}: HTTP {response.status_code}")
        return False

/Users/saakethkesireddy/Desktop/job_journal/resume_model/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [21]:
download_reddit_image(sample_url, "sample_image.jpg")

True

In [38]:
relevant_cohort["flair"].value_counts()

flair
Question                                    253
I Have a Question                           139
Discussion                                  127
Technology/Software/IT                       54
I’m Sharing Advice                           14
Engineering                                  12
Removed: Rule 6 - Use Proper Title/Flair      9
Success Story                                 9
I’m giving advice                             9
General/Other Industries                      6
Finance/Banking                               6
Marketing/Sales                               5
Healthcare/Medical                            3
Academia                                      2
Retail/Customer Service                       2
Creative/Media                                2
Manufacturing/Operations                      1
Human Resources                               1
Name: count, dtype: int64

In [24]:
relevant_cohort["selftext"].str.contains("advice")

20       False
25        True
44       False
86       False
92       False
         ...  
10075    False
10116    False
10117    False
10160    False
10172    False
Name: selftext, Length: 654, dtype: bool

In [25]:
import re
from collections import Counter
import pandas as pd

# 1. Filter out empty, removed, or deleted selftext
valid_texts = df_sample_posts["selftext"].dropna()
valid_texts = valid_texts[~valid_texts.isin(["[removed]", "[deleted]"])]

# 2. Extract all lowercase words across all descriptions
# \b[a-zA-Z0-9_\'-]+\b matches words and contractions (e.g., "i'm", "resume-focused")
all_words = re.findall(r"\b[a-zA-Z0-9_\'-]+\b", " ".join(valid_texts).lower())

# 3. Count frequencies
word_counts = Counter(all_words)

# 4. Convert to a DataFrame sorted by frequency
df_word_freq = pd.DataFrame(word_counts.most_common(), columns=["word", "count"])

print(df_word_freq.head(20))

      word  count
0        i  23634
1       to  15852
2      and  13817
3      the  13205
4        a  13192
5       my  11348
6      for   8583
7       in   8216
8       of   6241
9       it   6033
10  resume   5861
11      is   5758
12    that   4691
13      on   4657
14    have   4577
15      or   4397
16     but   4252
17     i'm   3683
18    with   3315
19      as   3156


In [26]:
# Common filler words to exclude
STOP_WORDS = {
    "i", "to", "and", "the", "a", "my", "for", "in", "of", "it", "is",
    "that", "on", "have", "or", "but", "with", "as", "this", "be", "so", 
    "at", "are", "if", "not", "any", "me", "been", "was", "from", "you",
    "an", "all", "would", "do", "what", "like", "just", "about", "can"
}

# Remove URLs first
cleaned_text = re.sub(r"https?://\S+", "", " ".join(valid_texts).lower())

# Extract words and filter
words = [w for w in re.findall(r"\b[a-z]{2,}\b", cleaned_text) if w not in STOP_WORDS]
df_top_keywords = pd.DataFrame(Counter(words).most_common(), columns=["keyword", "count"])

print(df_top_keywords.head(20))

       keyword  count
0       resume   5874
1          job   3099
2   experience   2977
3           am   2871
4           ve   2697
5        roles   2296
6         work   2015
7      looking   1927
8     applying   1897
9     feedback   1786
10         get   1678
11        some   1670
12        jobs   1519
13      should   1491
14        more   1469
15        time   1469
16         one   1469
17         out   1456
18         how   1431
19       years   1364
